# Exploring Embedding Models in LangChain

## Install OpenAI, HuggingFace and LangChain dependencies

In [1]:
!pip install langchain==1.2.12
!pip install langchain-openai==1.1.11
!pip install langchain-community==0.4.1
!pip install langchain-huggingface==1.2.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.1 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


## Enter Open AI and HuggingFace API Tokens

In [2]:
from getpass import getpass

OPENAI_KEY = getpass('Enter Open AI API Key: ')

Enter Open AI API Key: ··········


In [3]:
from getpass import getpass

HUGGINGFACEHUB_API_TOKEN = getpass('Enter HuggingFace Auth Token Key: ')

Enter HuggingFace Auth Token Key: ··········


## Setup Environment Variables

In [4]:
import os

os.environ['OPENAI_API_KEY'] = OPENAI_KEY
os.environ['HUGGINGFACEHUB_API_TOKEN'] = HUGGINGFACEHUB_API_TOKEN

## Embedding models

The Embeddings class is a class designed for interfacing with text embedding models. There are lots of embedding model providers (OpenAI, Cohere, Hugging Face, etc) - this class is designed to provide a standard interface for all of them.

Embeddings create a vector representation of a piece of text. This is useful because it means we can think about text in the vector space, and do things like semantic search where we look for pieces of text that are most similar in the vector space.

The base Embeddings class in LangChain provides two methods: one for embedding documents and one for embedding a query. The former takes as input multiple texts, while the latter takes a single text. The reason for having these as two separate methods is that some embedding providers have different embedding methods for documents (to be searched over) vs queries (the search query itself).

In [5]:
docs = [
    "cats eat and sleep",
    "dogs eat and bark",
    "cars drive fast",
    "vehicles include trucks and cars"
]

Embeddings create a vector representation of a piece of text. This is useful because it means we can think about text in the vector space, and do things like semantic search where we look for pieces of text that are most similar in the vector space.

The base Embeddings class in LangChain provides two methods: one for embedding documents and one for embedding a query. The former, `.embed_documents`, takes as input multiple texts, while the latter, `.embed_query`, takes a single text. The reason for having these as two separate methods is that some embedding providers have different embedding methods for documents (to be searched over) vs queries (the search query itself).

- `.embed_query`  will return a list of floats,
- `.embed_documents` returns a list of lists of floats.

### Open AI Embedding Models

LangChain enables us to access Open AI embedding models which include the newest models: a smaller and highly efficient `text-embedding-3-small` model, and a larger and more powerful `text-embedding-3-large` model.

In [6]:
from langchain_openai import OpenAIEmbeddings

# details here: https://openai.com/blog/new-embedding-models-and-api-updates
openai_embed_model = OpenAIEmbeddings(model='text-embedding-3-small')

In [7]:
embeddings = openai_embed_model.embed_documents(docs)

In [8]:
len(embeddings)

4

In [9]:
len(embeddings[0])

1536

In [10]:
print(embeddings[0])

[-0.0141448974609375, -0.004940032958984375, -0.00901031494140625, 0.01161956787109375, 0.020263671875, 0.026702880859375, 0.023834228515625, 0.035064697265625, -0.00505828857421875, 0.0032215118408203125, 0.0501708984375, 0.001251220703125, 0.031097412109375, 0.046966552734375, 0.003444671630859375, 0.01678466796875, 0.00811767578125, 0.01218414306640625, -0.0069732666015625, 0.0355224609375, 0.037261962890625, -0.0289764404296875, 0.0321044921875, -0.03582763671875, -0.0297088623046875, 0.05474853515625, -0.00849151611328125, 0.033538818359375, 0.01213836669921875, -0.0293121337890625, 0.0248260498046875, -0.040679931640625, -0.01139068603515625, -0.040863037109375, 0.01140594482421875, 0.023101806640625, 0.0202484130859375, -0.031494140625, -0.006778717041015625, 0.0133056640625, -0.0093841552734375, 0.01476287841796875, 0.0218658447265625, -0.00946044921875, -0.00905609130859375, -0.022430419921875, -0.0290374755859375, -0.00665283203125, 0.05450439453125, 0.021759033203125, -0.001

In [11]:
docs

['cats eat and sleep',
 'dogs eat and bark',
 'cars drive fast',
 'vehicles include trucks and cars']

In [12]:
from sklearn.metrics.pairwise import cosine_similarity

sim_matrix = cosine_similarity(embeddings)
sim_matrix

array([[1.        , 0.61268936, 0.20348487, 0.19823494],
       [0.61268936, 1.        , 0.26188947, 0.2003346 ],
       [0.20348487, 0.26188947, 1.        , 0.39907096],
       [0.19823494, 0.2003346 , 0.39907096, 1.        ]])

## Open Source Embedding Models on HuggingFace

`langchain-huggingface` integrates seamlessly with LangChain, providing an efficient and effective way to utilize Hugging Face models within the LangChain ecosystem.

`HuggingFaceEmbeddings`uses `sentence-transformers` embeddings. It computes the embedding locally, using your computer resources and allows you to access open or open source embedding LLMs hosted on HuggingFace.

In [13]:
from langchain_huggingface import HuggingFaceEmbeddings

# check out model details here: https://huggingface.co/mixedbread-ai/mxbai-embed-large-v1
model_name = "mixedbread-ai/mxbai-embed-large-v1"

hf_embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/266 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/677 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/670M [00:00<?, ?B/s]

... (2 outputs left out)

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

In [14]:
embeddings = hf_embeddings.embed_documents(docs)

In [15]:
len(embeddings)

4

In [16]:
len(embeddings[0])

1024

In [17]:
docs

['cats eat and sleep',
 'dogs eat and bark',
 'cars drive fast',
 'vehicles include trucks and cars']

In [18]:
sim_matrix = cosine_similarity(embeddings)
sim_matrix

array([[1.        , 0.52532456, 0.34220057, 0.33687413],
       [0.52532456, 1.        , 0.31717128, 0.33062157],
       [0.34220057, 0.31717128, 1.        , 0.72249644],
       [0.33687413, 0.33062157, 0.72249644, 1.        ]])

## Build a small search engine!

### Load Knowledgebase documents

In [19]:
documents = [
    'Quantum mechanics describes the behavior of very small particles.',
    'Photosynthesis is the process by which green plants make food using sunlight.',
    "Shakespeare's plays are a testament to English literature.",
    'Artificial Intelligence aims to create machines that can think and learn.',
    'The pyramids of Egypt are historical monuments that have stood for thousands of years.',
    'Biology is the study of living organisms and their interactions with the environment.',
    'Music therapy can aid in the mental well-being of individuals.',
    'The Milky Way is just one of billions of galaxies in the universe.',
    'Economic theories help understand the distribution of resources in society.',
    'Yoga is an ancient practice that involves physical postures and meditation.'
]

In [20]:
len(documents)

10

### Get document embeddings

In [21]:
document_embeddings = openai_embed_model.embed_documents(documents)

### Let's try to find the most similar document for one query

In [22]:
new_text = 'What is AI?'
new_text

'What is AI?'

In [23]:
query_embedding = openai_embed_model.embed_query(new_text)

In [24]:
cosine_similarities = cosine_similarity([query_embedding], document_embeddings)
cosine_similarities

array([[ 0.10208878,  0.09257072, -0.00536246,  0.63139091,  0.02338266,
         0.09308614,  0.10768831,  0.07000022,  0.05819709,  0.06607333]])

In [25]:
import numpy as np

documents[np.argmax(cosine_similarities[0])]

'Artificial Intelligence aims to create machines that can think and learn.'

### Create Search Engine function

In [26]:
def semantic_search_engine(query, embedder_model):
  query_embedding = embedder_model.embed_query(query)
  cos_scores = cosine_similarity([query_embedding], document_embeddings)[0]
  top_result_id = np.argmax(cos_scores)
  return documents[top_result_id]

### Try out the function

In [27]:
new_sentence = 'Tell me about AI'
semantic_search_engine(new_sentence, openai_embed_model)

'Artificial Intelligence aims to create machines that can think and learn.'

In [28]:
new_sentence = 'Do you know about the pyramids?'
semantic_search_engine(new_sentence, openai_embed_model)

'The pyramids of Egypt are historical monuments that have stood for thousands of years.'

In [29]:
new_sentence = 'How do plants survive?'
semantic_search_engine(new_sentence, openai_embed_model)

'Photosynthesis is the process by which green plants make food using sunlight.'